# Reward 模型

CartPole、LunarLander 等游戏里面他的奖励由环境反馈

但是在具体的LLM问题中，当前决策的好坏并没有直接的反馈， 因此需要一个Reward模型反馈

# **Badly Badly Terror 模型**

对于一个问题 \(x\)，假设有两个回答：

- \(y_w\)：更好的回答，即 winner；
- \(y_l\)：较差的回答，即 loser。

奖励模型 \(r(x,y)\) 会针对问题 \(x\) 和回答 \(y\) 输出一个实数分数。我们希望：

$$
r(x,y_w)>r(x,y_l).
$$

Bradley–Terry 模型将“回答 \(y_w\) 优于回答 \(y_l\)”的概率定义为：

$$
P(y_w>y_l\mid x)
=\sigma\left(r(x,y_w)-r(x,y_l)\right),
$$

其中 Sigmoid 函数为：

$$
\sigma(z)=\frac{1}{1+e^{-z}}.
$$

---

**为什么使用减法？**

定义两个回答的奖励差：

$$
\Delta r=r(x,y_w)-r(x,y_l).
$$

模型只需要判断两个回答的相对优劣，而不需要关心奖励分数的绝对大小：

- 当 \(\Delta r>0\) 时，模型认为 \(y_w\) 更好；
- 当 \(\Delta r=0\) 时，模型认为两个回答同样好；
- 当 \(\Delta r<0\) 时，模型错误地认为 \(y_l\) 更好。

使用差值还有一个重要性质：同时给两个回答的奖励增加同一个常数 \(C\)，比较结果不会发生变化：

$$
\begin{aligned}
&[r(x,y_w)+C]-[r(x,y_l)+C]\\
={}&r(x,y_w)-r(x,y_l).
\end{aligned}
$$

因此，Bradley–Terry 模型学习的是回答之间的**相对偏好**，而不是奖励分数的绝对标尺。

---

**为什么使用 Sigmoid？**

奖励差 \(\Delta r\) 的取值范围是整个实数轴：

$$
\Delta r\in(-\infty,+\infty).
$$

Sigmoid 可以将它转换为 \((0,1)\) 之间的概率：

$$
\sigma(\Delta r)\in(0,1).
$$

具体来说：

$$
\begin{cases}
\Delta r\gg 0, & \sigma(\Delta r)\approx 1,\\
\Delta r=0, & \sigma(\Delta r)=0.5,\\
\Delta r\ll 0, & \sigma(\Delta r)\approx 0.
\end{cases}
$$

因此：

- 奖励差越大，模型认为 \(y_w\) 更好的概率越高；
- 奖励差为零，模型认为两个回答各有 \(50\%\) 的胜率；
- 奖励差为负，模型认为 \(y_w\) 更好的概率低于 \(50\%\)。

Sigmoid 还是一个单调、连续且可微的函数，因此适合通过梯度下降训练神经网络。

> Sigmoid 并不意味着模型“完全不受数值大小影响”。当输入的绝对值很大时，Sigmoid 会逐渐饱和，梯度也会变小。它的主要作用是将奖励差解释成概率，并提供可微的优化目标。

---

**奖励模型的损失函数**

我们希望偏好数据中标记的 winner \(y_w\) 获得尽可能高的胜出概率：

$$
P(y_w>y_l\mid x)
=\sigma(\Delta r).
$$

因此可以使用最大似然估计，最大化：

$$
\log\sigma(\Delta r).
$$

训练通常采用梯度下降，所以将最大化问题改写为最小化负对数似然：

$$
\boxed{
\mathcal L
=-\log\sigma\left(r(x,y_w)-r(x,y_l)\right)
}
$$

这里不是简单地在 Sigmoid 前面加负号，而是使用：

$$
-\log\sigma(\Delta r).
$$

原因是：

- 当 \(\Delta r\) 很大时，\(\sigma(\Delta r)\approx1\)，损失接近 \(0\)；
- 当 \(\Delta r=0\) 时，\(\sigma(\Delta r)=0.5\)，损失为 \(-\log 0.5\)；
- 当 \(\Delta r\) 很小时，\(\sigma(\Delta r)\approx0\)，损失会变得很大。

因此，最小化这个损失会推动：

$$
r(x,y_w)-r(x,y_l)
$$

不断增大，也就是推动模型满足：

$$
r(x,y_w)>r(x,y_l).
$$